# Validation: LLM Responses without XAI (N_SAMPLES=10)

This notebook validates LLM analysis outputs for the without-XAI experiment with 10 training/prediction samples.

## Validation Goals

1. **Feature Ranking Accuracy**: Compare LLM's ranked features vs ground truth SHAP rankings
2. **SHAP Citation Accuracy**: Check if LLMs correctly cite SHAP values (when they do)
3. **Fabrication Detection**: Identify claims not supported by the data
4. **Cybersecurity Insight Quality**: Assess practical SOC relevance

In [1]:
import json
import re
from collections import defaultdict

# Load results
with open('resultados_without_xai_samples_10.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

print(f"Loaded {len(results)} model responses")
print(f"Models: {list(results.keys())}")

Loaded 4 model responses
Models: ['glm-4.7-flash:latest', 'qwen3:14b', 'gpt-oss:20b', 'qwen3:30b']


## Ground Truth Reference

From the actual SHAP analysis of the Random Forest model:

In [2]:
# Ground truth SHAP rankings (mean |SHAP| value)
GROUND_TRUTH = {
    'BotAttack': ['Port', 'Status', 'Payload_Size'],
    'Normal': ['Port', 'Status', 'Payload_Size'],
    'PortScan': ['Payload_Size', 'Status', 'Port'],
}

OVERALL_IMPORTANCE = ['Port', 'Status', 'Payload_Size', 'Request_Type', 'Protocol', 'User_Agent']

print("Ground Truth SHAP Rankings:")
for cls, features in GROUND_TRUTH.items():
    print(f"  {cls}: {features}")

print(f"\nOverall feature importance: {OVERALL_IMPORTANCE}")
print("\nKey insight: User_Agent has near-zero SHAP importance (<0.006) despite being commonly overvalued by LLMs")

Ground Truth SHAP Rankings:
  BotAttack: ['Port', 'Status', 'Payload_Size']
  Normal: ['Port', 'Status', 'Payload_Size']
  PortScan: ['Payload_Size', 'Status', 'Port']

Overall feature importance: ['Port', 'Status', 'Payload_Size', 'Request_Type', 'Protocol', 'User_Agent']

Key insight: User_Agent has near-zero SHAP importance (<0.006) despite being commonly overvalued by LLMs


## Validation Functions

In [3]:
def extract_feature_rankings(text):
    """Extract feature importance rankings from LLM text."""
    features = ['Port', 'Status', 'Payload_Size', 'Request_Type', 'Protocol', 'User_Agent']
    feature_mentions = defaultdict(int)
    
    for feat in features:
        importance_contexts = ['important', 'significant', 'key', 'main', 'primary', 'drives', 'influences', 'impact']
        for ctx in importance_contexts:
            pattern = feat + r'[^.]*' + ctx + r'[^.]|' + ctx + r'[^.]' + feat
            if re.search(pattern, text, re.IGNORECASE):
                feature_mentions[feat] += 1
    
    rankings = {}
    text_lower = text.lower()
    for feat in features:
        if any(phrase in text_lower for phrase in [f'{feat.lower()} most', f'most {feat.lower()}', f'primary {feat.lower()}']):
            rankings[feat] = 'top'
    
    return dict(feature_mentions), rankings


def check_shap_citation(text):
    """Check if LLM mentions SHAP/LIME (inappropriate in without-XAI context)."""
    return {
        'has_shap': bool(re.search(r'shap', text, re.IGNORECASE)),
        'has_lime': bool(re.search(r'lime', text, re.IGNORECASE)),
        'inappropriate_xai_citation': bool(re.search(r'shap|lime', text, re.IGNORECASE))
    }


def detect_fabrication(text):
    """Detect potential fabrications."""
    fabrications = []
    metric_claims = re.findall(r'(?:precision|recall|f1|f-score)\s*[=:]\s*[\d.]+', text, re.IGNORECASE)
    if metric_claims:
        fabrications.append(('Per-class metrics claimed', metric_claims))
    shap_values = re.findall(r'shap value[s]?\s*[=:]\s*[\d.]+', text, re.IGNORECASE)
    if shap_values:
        fabrications.append(('SHAP values claimed', shap_values))
    return fabrications


def validate_response(model_name, text):
    """Run all validations on a response."""
    mentions, rankings = extract_feature_rankings(text)
    xai_check = check_shap_citation(text)
    fabrications = detect_fabrication(text)
    
    user_agent_overvalued = mentions.get("User_Agent", 0) > 0 or "User_Agent" in rankings
    top3_mentioned = sorted(mentions.items(), key=lambda x: -x[1])[:3]
    top3_features = [f[0] for f in top3_mentioned]
    
    has_port_top = mentions.get("Port", 0) == max(mentions.values()) if mentions else False
    has_correct_top3 = 'Port' in top3_features and 'Status' in top3_features
    
    return {
        'model': model_name,
        'char_count': len(text),
        'feature_mentions': mentions,
        'top3_features': top3_features,
        'user_agent_overvalued': user_agent_overvalued,
        'xai_citation': xai_check,
        'fabrications': fabrications,
        'has_port_top': has_port_top,
        'has_correct_top3': has_correct_top3
    }

## Run Validation

In [4]:
# Validate all models
validations = {}
for model, text in results.items():
    print(f"\n{'='*60}")
    print(f"Validating: {model}")
    print(f"{'='*60}")
    validations[model] = validate_response(model, text)
    
    print(f"Character count: {validations[model]['char_count']}")
    print(f"Top-3 features mentioned: {validations[model]['top3_features']}")
    print(f"User_Agent overvalued: {validations[model]['user_agent_overvalued']}")
    print(f"XAI citation (inappropriate): {validations[model]['xai_citation']}")
    print(f"Fabrications: {validations[model]['fabrications']}")


Validating: glm-4.7-flash:latest
Character count: 6533
Top-3 features mentioned: []
User_Agent overvalued: False
XAI citation (inappropriate): {'has_shap': True, 'has_lime': False, 'inappropriate_xai_citation': True}
Fabrications: []

Validating: qwen3:14b
Character count: 5915
Top-3 features mentioned: []
User_Agent overvalued: False
XAI citation (inappropriate): {'has_shap': True, 'has_lime': True, 'inappropriate_xai_citation': True}
Fabrications: []

Validating: gpt-oss:20b
Character count: 8765
Top-3 features mentioned: ['Port']
User_Agent overvalued: False
XAI citation (inappropriate): {'has_shap': True, 'has_lime': True, 'inappropriate_xai_citation': True}
Fabrications: []

Validating: qwen3:30b
Character count: 6977
Top-3 features mentioned: ['Port']
User_Agent overvalued: False
XAI citation (inappropriate): {'has_shap': True, 'has_lime': False, 'inappropriate_xai_citation': True}
Fabrications: []


## Summary Table

In [5]:
import pandas as pd

summary_data = []
for model, val in validations.items():
    summary_data.append({
        'Model': model,
        'Response Length': val['char_count'],
        'Port as Top Feature': val['has_port_top'],
        'Correct Top-3 (Port+Status)': val['has_correct_top3'],
        'User_Agent Overvalued': val['user_agent_overvalued'],
        'Inappropriate XAI Citation': val['xai_citation']['inappropriate_xai_citation'],
        'Fabrications Detected': len(val['fabrications']) > 0
    })

df_summary = pd.DataFrame(summary_data)
print("\nValidation Summary (N_SAMPLES=10):")
print(df_summary.to_string(index=False))


Validation Summary (N_SAMPLES=10):
               Model  Response Length  Port as Top Feature  Correct Top-3 (Port+Status)  User_Agent Overvalued  Inappropriate XAI Citation  Fabrications Detected
glm-4.7-flash:latest             6533                False                        False                  False                        True                  False
           qwen3:14b             5915                False                        False                  False                        True                  False
         gpt-oss:20b             8765                 True                        False                  False                        True                  False
           qwen3:30b             6977                 True                        False                  False                        True                  False


## Key Findings

In [6]:
print("\n" + "="*60)
print("KEY FINDINGS")
print("="*60)

correct_top3_count = sum(1 for v in validations.values() if v['has_correct_top3'])
user_agent_bias_count = sum(1 for v in validations.values() if v['user_agent_overvalued'])

print(f"\n1. Feature Ranking Accuracy:")
print(f"   - Models with Port+Status in top-3: {correct_top3_count}/4")
    
print(f"\n2. Known Bias (User_Agent overvaluation):")
print(f"   - Models showing User_Agent bias: {user_agent_bias_count}/4")
    
print(f"\n3. XAI Citations (should be 0 in without-XAI):")
xai_count = sum(1 for v in validations.values() if v['xai_citation']['inappropriate_xai_citation'])
print(f"   - Models with inappropriate XAI citations: {xai_count}/4")
    
print(f"\n4. Fabrications:")
fabrication_count = sum(1 for v in validations.values() if len(v['fabrications']) > 0)
print(f"   - Models with fabrications: {fabrication_count}/4")


KEY FINDINGS

1. Feature Ranking Accuracy:
   - Models with Port+Status in top-3: 0/4

2. Known Bias (User_Agent overvaluation):
   - Models showing User_Agent bias: 0/4

3. XAI Citations (should be 0 in without-XAI):
   - Models with inappropriate XAI citations: 4/4

4. Fabrications:
   - Models with fabrications: 0/4


## Feature Mention Analysis

In [7]:
print("\nFeature Mention Analysis:")
for model, val in validations.items():
    print(f"\n{model}:")
    for feat, count in sorted(val['feature_mentions'].items(), key=lambda x: -x[1]):
        print(f"  {feat}: {count}")


Feature Mention Analysis:

glm-4.7-flash:latest:

qwen3:14b:

gpt-oss:20b:
  Port: 1

qwen3:30b:
  Port: 1
